<a href="https://colab.research.google.com/github/mikkaK/ZC3-team-generator/blob/main/ZC3_Team_generator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [98]:
import csv
import re
from datetime import datetime

#CSV_PATH = "/content/ZC3_Company_Triathlon_28.06.csv"
#CSV_PATH = "/content/zc3_edgecases_conflicts.csv"
CSV_PATH = "/content/zc3_edgecases_valid.csv"

ROLES = ["swimmer", "biker", "runner"]

def _norm(s: str) -> str:
    return (s or "").strip()

def _norm_lower(s: str) -> str:
    return _norm(s).lower()

def _pick(row: dict, name: str) -> str:
    if name in row:
        return row.get(name, "") or ""
    m = {k.lower(): k for k in row.keys()}
    k = m.get(name.lower())
    return (row.get(k, "") or "") if k else ""

def parse_completion_time(raw: str):
    raw = _norm(raw)
    if not raw:
        return None
    # Your CSV example: "1/30/26 13:15"
    for fmt in ("%m/%d/%y %H:%M", "%m/%d/%Y %H:%M", "%Y-%m-%d %H:%M:%S", "%Y-%m-%d %H:%M"):
        try:
            return datetime.strptime(raw, fmt)
        except ValueError:
            pass
    return None

def parse_distance(raw: str) -> str:
    r = _norm_lower(raw)
    if "both" in r:
        return "both"
    if "sprint" in r:
        return "sprint"
    if "olympic" in r:
        return "olympic"
    return "both"

def parse_ambition(raw: str) -> str:
    r = _norm_lower(raw)
    if "competitive" in r:
        return "competitive"
    if "fun" in r:
        return "fun"
    if "both" in r:
        return "both"
    return "both"

def parse_gender(raw: str) -> str:
    r = _norm_lower(raw)
    if r in ("man", "m", "male"):
        return "m"
    if r in ("woman", "w", "f", "female"):
        return "f"
    return "x"

def parse_prio(raw: str):
    r = _norm_lower(raw).replace(" ", "")
    if "nogo" in r or "no-go" in r:
        return "nogo"
    m = re.search(r"prio(\d)", r)
    return int(m.group(1)) if m else None

def load_registrants(csv_path: str) -> list[dict]:
    regs = []
    with open(csv_path, "r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            completed_at = parse_completion_time(_pick(row, "Completion time"))

            # Use Company Email if present, else fallback to Email; normalize to lowercase
            email = _norm_lower(_pick(row, "Company Email (firstname.name@accenture.com)")) or _norm_lower(_pick(row, "Email"))
            if not email:
                continue

            # teammate emails (normalized)
            tm_swim = _norm_lower(_pick(row, "Swimmer (Company Email)"))
            tm_bike = _norm_lower(_pick(row, "Biker (Company Email)"))
            tm_run  = _norm_lower(_pick(row, "Runner (Company Email)"))

            pr_swim = parse_prio(_pick(row, "Priority.Swimmer"))
            pr_bike = parse_prio(_pick(row, "Priority.Biker"))
            pr_run  = parse_prio(_pick(row, "Priority.Runner"))

            prios = {"swimmer": pr_swim, "biker": pr_bike, "runner": pr_run}
            prio_by_rank = {1: None, 2: None, 3: None}
            nogo = []

            for role, pr in prios.items():
                if pr == "nogo":
                    nogo.append(role)
                elif isinstance(pr, int) and pr in (1, 2, 3) and prio_by_rank[pr] is None:
                    prio_by_rank[pr] = role

            remaining = [r for r in ROLES if (r not in nogo and r not in prio_by_rank.values())]
            for rank in (1, 2, 3):
                if prio_by_rank[rank] is None and remaining:
                    prio_by_rank[rank] = remaining.pop(0)

            regs.append({
                "id": email,
                "name": _norm(_pick(row, "Name")),
                "completed_at": completed_at,

                "gender": parse_gender(_pick(row, "Gender")),
                "distance": parse_distance(_pick(row, "Preferred distance")),
                "ambition": parse_ambition(_pick(row, "Ambition")),
                "remarks": _norm(_pick(row, "Remarks")),

                "prio1": prio_by_rank[1] or "swimmer",
                "prio2": prio_by_rank[2] or "biker",
                "prio3": prio_by_rank[3] or "runner",
                "nogo": nogo,

                "requested_teammates": {
                    "swimmer": tm_swim,
                    "biker": tm_bike,
                    "runner": tm_run,
                },
            })

    # Sort chronologically (earliest first). Missing timestamps go last.
    regs.sort(key=lambda r: (r["completed_at"] is None, r["completed_at"]))
    return regs

registrants = load_registrants(CSV_PATH)
print("Loaded:", len(registrants))
print("First 3 (chronological):")
for r in registrants[:3]:
    print(r["completed_at"], r["id"], r["distance"], r["ambition"])

Loaded: 60
First 3 (chronological):
2026-06-28 08:00:00 person001@accenture.com olympic competitive
2026-06-28 08:05:00 person002@accenture.com olympic competitive
2026-06-28 08:10:00 person003@accenture.com olympic competitive


In [99]:
# ---- Build hard custom groups via Union-Find, with role locks ----

class UnionFind:
    def __init__(self):
        self.parent = {}
    def find(self, x):
        if x not in self.parent:
            self.parent[x] = x
        if self.parent[x] != x:
            self.parent[x] = self.find(self.parent[x])
        return self.parent[x]
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb:
            self.parent[rb] = ra

def build_units(registrants: list[dict]):
    by_id = {r["id"]: r for r in registrants}
    uf = UnionFind()
    issues = []

    for r in registrants:
        uf.find(r["id"])

    # must-link edges
    for r in registrants:
        me = r["id"]
        for role, mate in r["requested_teammates"].items():
            if mate:
                if mate not in by_id:
                    issues.append(f"Teammate not found: {me} requested {role}={mate}")
                    continue
                uf.union(me, mate)

    # groups
    groups = {}
    for pid in by_id:
        root = uf.find(pid)
        groups.setdefault(root, []).append(pid)

    # role locks: if A enters runner=B => B locked to runner in that group
    role_locks_by_group = {root: {} for root in groups}
    for r in registrants:
        root = uf.find(r["id"])
        locks = role_locks_by_group[root]
        for role, mate in r["requested_teammates"].items():
            if mate and mate in by_id:
                if mate in locks and locks[mate] != role:
                    issues.append(f"ROLE CONFLICT: {mate} locked as {locks[mate]} and {role}")
                locks[mate] = role

    used_in_group = set()
    units = []

    for root, member_ids in groups.items():
        if len(member_ids) > 3:
            issues.append(f"GROUP TOO LARGE (>3): {root} has {len(member_ids)} members: {member_ids}")
            continue

        members = [by_id[mid] for mid in member_ids]
        used_in_group.update(member_ids)

        # unit time = earliest completion among members
        times = [m["completed_at"] for m in members if m["completed_at"] is not None]
        unit_time = min(times) if times else None

        units.append({
            "unit_id": root,
            "members": members,
            "role_locks": role_locks_by_group[root],  # member_id -> locked_role
            "is_custom": True,
            "unit_time": unit_time,
        })

    # singles not in any group
    for r in registrants:
        if r["id"] not in used_in_group:
            units.append({
                "unit_id": r["id"],
                "members": [r],
                "role_locks": {},
                "is_custom": False,
                "unit_time": r["completed_at"],
            })

    # chronological ordering (missing timestamps last)
    units.sort(key=lambda u: (u["unit_time"] is None, u["unit_time"]))

    return units, issues

units, issues = build_units(registrants)
print("Units:", len(units))
if issues:
    print("\nISSUES (first 30):")
    for x in issues[:30]:
        print(" -", x)

Units: 56


In [100]:
import itertools
import csv

TEAM_PLAN = [("olympic", 7), ("sprint", 8)]  # exactly 7 olympic teams + 8 sprint teams

def ambition_hard_ok(people):
    ambs = {p["ambition"] for p in people}
    return not ("competitive" in ambs and "fun" in ambs)

def choose_team_distance(people):
    fixed = set()
    for p in people:
        d = p["distance"]
        if d in ("sprint", "olympic"):
            fixed.add(d)
    if len(fixed) == 0:
        return "free"
    if len(fixed) == 1:
        return next(iter(fixed))
    return None  # impossible

def role_allowed(person, role):
    return role not in set(person.get("nogo", []))

def role_score(person, role):
    if not role_allowed(person, role):
        return -10**9
    if role == person["prio1"]: return 30
    if role == person["prio2"]: return 20
    if role == person["prio3"]: return 10
    return 1

def best_role_assignment(people, role_locks):
    ids = [p["id"] for p in people]
    locked_roles = [role_locks[i] for i in ids if i in role_locks]
    if len(set(locked_roles)) != len(locked_roles):
        return None, None

    best = None
    best_map = None
    for perm in itertools.permutations(ROLES, len(people)):
        ok = True
        total = 0
        m = {}
        for p, role in zip(people, perm):
            pid = p["id"]
            if pid in role_locks and role_locks[pid] != role:
                ok = False; break
            s = role_score(p, role)
            if s < -10**8:
                ok = False; break
            total += s
            m[pid] = role
        if ok and (best is None or total > best):
            best, best_map = total, m
    return best, best_map

def unit_distance_eligibility(unit, target_dist):
    # olympic: members must be olympic or both
    # sprint: members must be sprint or both
    allowed = {target_dist, "both"}
    return all(m["distance"] in allowed for m in unit["members"])

def unit_is_both_only(unit):
    return all(m["distance"] == "both" for m in unit["members"])

def team_score(people, role_locks, target_dist):
    # hard ambition
    if not ambition_hard_ok(people):
        return -10**9, None

    # hard distance
    if any(p["distance"] not in (target_dist, "both") for p in people):
        return -10**9, None

    # role feasibility
    rs, role_map = best_role_assignment(people, role_locks)
    if role_map is None:
        return -10**9, None

    # ambition preference (strong)
    ambs = [p["ambition"] for p in people]
    if all(a == "competitive" for a in ambs): amb_score = 200
    elif ("competitive" in ambs and "fun" not in ambs): amb_score = 140
    elif all(a == "fun" for a in ambs): amb_score = 120
    elif ("fun" in ambs and "competitive" not in ambs): amb_score = 90
    else: amb_score = 0

    # joker penalty
    joker_pen = sum(3 for p in people if p["distance"] == "both")

    return rs + amb_score - joker_pen, role_map

def pick_next_team(units_available, target_dist):
    """
    Build one team of 3 people using chronological priority:
    - anchor = earliest eligible unit
    - then dynamically widen search window until feasible completion is found
    """
    eligible = [u for u in units_available if unit_distance_eligibility(u, target_dist)]
    if not eligible:
        return None, None, None

    # sprint phase rule: prioritize sprint signups first, then both
    if target_dist == "sprint":
        eligible.sort(key=lambda u: (unit_is_both_only(u), u["unit_time"] is None, u["unit_time"]))
    else:
        eligible.sort(key=lambda u: (u["unit_time"] is None, u["unit_time"]))

    anchor = eligible[0]
    anchor_size = len(anchor["members"])

    # try growing window
    window = 30
    step = 25

    while True:
        cand = eligible[:min(window, len(eligible))]

        # Must include anchor
        others = [u for u in cand if u["unit_id"] != anchor["unit_id"]]

        best = None
        best_choice = None
        best_roles = None

        # combinations depending on anchor size
        if anchor_size == 3:
            people = anchor["members"]
            score, roles = team_score(people, dict(anchor["role_locks"]), target_dist)
            if score > -10**8:
                return [anchor], roles, score

        elif anchor_size == 2:
            for u1 in others:
                if len(u1["members"]) != 1:
                    continue
                people = anchor["members"] + u1["members"]
                locks = dict(anchor["role_locks"])
                locks.update(u1["role_locks"])
                score, roles = team_score(people, locks, target_dist)
                if score > -10**8 and (best is None or score > best):
                    best, best_choice, best_roles = score, [anchor, u1], roles

        elif anchor_size == 1:
            # anchor + (size2) OR anchor + (size1+size1)
            for u1 in others:
                if len(u1["members"]) == 2:
                    people = anchor["members"] + u1["members"]
                    locks = dict(anchor["role_locks"]); locks.update(u1["role_locks"])
                    score, roles = team_score(people, locks, target_dist)
                    if score > -10**8 and (best is None or score > best):
                        best, best_choice, best_roles = score, [anchor, u1], roles

            singles = [u for u in others if len(u["members"]) == 1]
            for u1, u2 in itertools.combinations(singles, 2):
                people = anchor["members"] + u1["members"] + u2["members"]
                locks = dict(anchor["role_locks"]); locks.update(u1["role_locks"]); locks.update(u2["role_locks"])
                score, roles = team_score(people, locks, target_dist)
                if score > -10**8 and (best is None or score > best):
                    best, best_choice, best_roles = score, [anchor, u1, u2], roles

        if best_choice is not None:
            return best_choice, best_roles, best

        if window >= len(eligible):
            return None, None, None  # no feasible completion exists
        window = min(len(eligible), window + step)

def build_exact_teams(units, plan):
    units_available = units[:]
    final_teams = []

    for dist, team_count in plan:
        for _ in range(team_count):
            chosen_units, roles, score = pick_next_team(units_available, dist)
            if chosen_units is None:
                raise RuntimeError(f"Could not complete required team for {dist}. Try relaxing constraints or check data conflicts.")
            # remove chosen units from availability
            chosen_ids = {u["unit_id"] for u in chosen_units}
            units_available = [u for u in units_available if u["unit_id"] not in chosen_ids]

            members = []
            role_locks = {}
            for u in chosen_units:
                members.extend(u["members"])
                role_locks.update(u["role_locks"])

            final_teams.append({
                "distance": dist,
                "members": members,
                "roles": roles,
                "team_type": "CUSTOM" if any(u["is_custom"] for u in chosen_units) else "AUTO"
            })

    # waitlist = all remaining members from remaining units
    waitlist = []
    for u in units_available:
        waitlist.extend(u["members"])

    return final_teams, waitlist

final_teams, waitlist = build_exact_teams(units, TEAM_PLAN)

print("Teams built:", len(final_teams), "=> athletes:", sum(len(t["members"]) for t in final_teams))
print("Waitlist athletes:", len(waitlist))

# --- compatibility hint for waitlist ---
def waitlist_hint(person):
    # distance fit
    d = person["distance"]
    if d == "olympic": dist_hint = "olympic"
    elif d == "sprint": dist_hint = "sprint"
    else: dist_hint = "either (both)"
    # ambition fit
    a = person["ambition"]
    if a == "competitive": amb_hint = "competitive teams only"
    elif a == "fun": amb_hint = "fun teams only"
    else: amb_hint = "either (both)"
    # role pref
    return dist_hint, amb_hint, person["prio1"]

# --- export CSV ---
OUTPUT_CSV = "/content/zc3_team_assignment_result.csv"

rows = []
for team_idx, t in enumerate(final_teams, start=1):
    for m in t["members"]:
        rows.append({
            "team_id": team_idx,
            "team_type": t["team_type"],
            "team_distance": t["distance"],
            "completed_at": m["completed_at"].isoformat(sep=" ") if m["completed_at"] else "",
            "member_email": m["id"],
            "member_name": m.get("name",""),
            "gender": m.get("gender",""),
            "ambition": m.get("ambition",""),
            "preferred_distance": m.get("distance",""),
            "assigned_role": t["roles"].get(m["id"], ""),
            "waitlist": "NO",
            "waitlist_distance_hint": "",
            "waitlist_ambition_hint": "",
            "waitlist_best_role": "",
        })

for m in waitlist:
    dist_hint, amb_hint, best_role = waitlist_hint(m)
    rows.append({
        "team_id": "WAITLIST",
        "team_type": "WAITLIST",
        "team_distance": "",
        "completed_at": m["completed_at"].isoformat(sep=" ") if m["completed_at"] else "",
        "member_email": m["id"],
        "member_name": m.get("name",""),
        "gender": m.get("gender",""),
        "ambition": m.get("ambition",""),
        "preferred_distance": m.get("distance",""),
        "assigned_role": "",
        "waitlist": "YES",
        "waitlist_distance_hint": dist_hint,
        "waitlist_ambition_hint": amb_hint,
        "waitlist_best_role": best_role,
    })

with open(OUTPUT_CSV, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
    writer.writeheader()
    writer.writerows(rows)

print("Exported:", OUTPUT_CSV)

Teams built: 15 => athletes: 45
Waitlist athletes: 15
Exported: /content/zc3_team_assignment_result.csv


In [101]:
from openpyxl import Workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
from openpyxl.chart import BarChart, PieChart, Reference
from openpyxl.chart.label import DataLabelList
from collections import Counter

OUTPUT_XLSX = "/content/zc3_team_assignment_result.xlsx"

# ---------- Styling helpers ----------
def auto_width(ws, min_w=12, max_w=60):
    for col in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col[0].column)
        for cell in col:
            if cell.value is not None and cell.value != "":
                max_len = max(max_len, len(str(cell.value)))
        ws.column_dimensions[col_letter].width = max(min_w, min(max_w, max_len + 2))

def style_header(ws, row=1):
    header_fill = PatternFill("solid", fgColor="111827")  # near-black
    header_font = Font(bold=True, color="FFFFFF")
    for cell in ws[row]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"

thin = Side(style="thin", color="E5E7EB")  # light gray
border = Border(left=thin, right=thin, top=thin, bottom=thin)

def apply_table_borders(ws, min_row, max_row, min_col, max_col):
    for r in range(min_row, max_row + 1):
        for c in range(min_col, max_col + 1):
            ws.cell(r, c).border = border
            ws.cell(r, c).alignment = Alignment(vertical="center", wrap_text=True)

# ---------- Color system (minimal, consistent) ----------
FILL_OLY = PatternFill("solid", fgColor="E8F5E9")     # light green
FILL_SPR = PatternFill("solid", fgColor="E3F2FD")     # light blue
FILL_BOTH = PatternFill("solid", fgColor="F3F4F6")    # light gray

FILL_COMP = PatternFill("solid", fgColor="FFEBEE")    # light red
FILL_FUN  = PatternFill("solid", fgColor="FFF8E1")    # light yellow
FILL_AMB_BOTH = PatternFill("solid", fgColor="F3F4F6")# light gray

def fill_for_distance(d):
    d = (d or "").lower()
    if d == "olympic": return FILL_OLY
    if d == "sprint":  return FILL_SPR
    if d == "both":    return FILL_BOTH
    return FILL_BOTH

def fill_for_ambition(a):
    a = (a or "").lower()
    if a == "competitive": return FILL_COMP
    if a == "fun":         return FILL_FUN
    if a == "both":        return FILL_AMB_BOTH
    return FILL_AMB_BOTH

def iso_or_blank(dt):
    return dt.isoformat(" ") if dt else ""

# ---------- Build flat rows ----------
teams_rows = []
for team_idx, team in enumerate(final_teams, start=1):
    for m in team["members"]:
        teams_rows.append({
            "team_id": team_idx,
            "team_type": team.get("team_type", "AUTO"),
            "team_distance": (team.get("distance") or ""),
            "assigned_role": team["roles"].get(m["id"], ""),
            "member_name": m.get("name", ""),
            "member_email": m["id"],
            "gender": m.get("gender", ""),
            "ambition": m.get("ambition", ""),
            "preferred_distance": m.get("distance", ""),
            "completed_at": iso_or_blank(m.get("completed_at")),
        })

waitlist_rows = []
for m in waitlist:
    waitlist_rows.append({
        "completed_at": iso_or_blank(m.get("completed_at")),
        "email": m["id"],
        "name": m.get("name", ""),
        "gender": m.get("gender", ""),
        "ambition": m.get("ambition", ""),
        "preferred_distance": m.get("distance", ""),
        "best_role": m.get("prio1", ""),
        "distance_hint": ("sprint/olympic" if m.get("distance") == "both" else m.get("distance","")),
        "ambition_hint": ("fun/competitive" if m.get("ambition") == "both" else m.get("ambition","")),
    })

raw_rows = []
for r in registrants:
    raw_rows.append({
        "completed_at": iso_or_blank(r.get("completed_at")),
        "email": r["id"],
        "name": r.get("name", ""),
        "gender": r.get("gender", ""),
        "distance": r.get("distance", ""),
        "ambition": r.get("ambition", ""),
        "teammate_swimmer": r.get("requested_teammates", {}).get("swimmer", ""),
        "teammate_biker": r.get("requested_teammates", {}).get("biker", ""),
        "teammate_runner": r.get("requested_teammates", {}).get("runner", ""),
        "remarks": r.get("remarks", ""),
    })

# ---------- KPI calculations ----------
total_regs = len(registrants)
assigned_count = sum(len(t["members"]) for t in final_teams)
waitlist_count = len(waitlist)
teams_by_dist = Counter(t.get("distance","") for t in final_teams)
athletes_by_dist = Counter()
for tr in teams_rows:
    athletes_by_dist[(tr["team_distance"] or "").lower()] += 1

amb_dist = Counter((tr["team_distance"].lower(), tr["ambition"].lower()) for tr in teams_rows)
role_dist = Counter(tr["assigned_role"].lower() for tr in teams_rows)

# ---------- Waitlist widget: earliest candidate per (distance, role, ambition-category) ----------
def _dt_key(x):
    return (x.get("completed_at") is None, x.get("completed_at"))

waitlist_sorted = sorted(waitlist, key=_dt_key)

def top_waitlist_candidate(distance: str, role: str, ambition_cat: str):
    for m in waitlist_sorted:
        d = (m.get("distance") or "").lower()
        a = (m.get("ambition") or "").lower()
        pr1 = (m.get("prio1") or "").lower()
        if d not in (distance, "both"):
            continue
        if a != ambition_cat:
            continue
        if pr1 != role:
            continue
        return m
    return None

# ---------- Create workbook ----------
wb = Workbook()
wb.remove(wb.active)

# =========================
# Hidden helper sheet (data for charts)
# =========================
ws_data = wb.create_sheet("Dashboard_Data")
ws_data.sheet_state = "hidden"

# Ambition table for bar chart
ws_data["B2"] = "distance"
ws_data["C2"] = "competitive"
ws_data["D2"] = "fun"
ws_data["E2"] = "both"
dist_list = ["olympic", "sprint"]
for i, dist in enumerate(dist_list, start=3):
    ws_data[f"B{i}"] = dist
    ws_data[f"C{i}"] = amb_dist.get((dist,"competitive"), 0)
    ws_data[f"D{i}"] = amb_dist.get((dist,"fun"), 0)
    ws_data[f"E{i}"] = amb_dist.get((dist,"both"), 0)

# Role table for pie chart
ws_data["H2"] = "role"
ws_data["I2"] = "count"
role_order = ["swimmer", "biker", "runner"]
for idx, role in enumerate(role_order, start=3):
    ws_data[f"H{idx}"] = role
    ws_data[f"I{idx}"] = role_dist.get(role, 0)

# =========================
# Dashboard Sheet
# =========================
ws = wb.create_sheet("Dashboard")

# layout grid
ws.column_dimensions["A"].width = 3
ws.column_dimensions["B"].width = 28
ws.column_dimensions["C"].width = 28
ws.column_dimensions["D"].width = 28
ws.column_dimensions["E"].width = 28
ws.column_dimensions["F"].width = 28
ws.column_dimensions["G"].width = 3
ws.column_dimensions["H"].width = 22
ws.column_dimensions["I"].width = 14
ws.column_dimensions["J"].width = 3

# Title
ws["B2"] = "ZC3 Company Triathlon — Team Assignment Dashboard"
ws["B2"].font = Font(size=18, bold=True, color="111827")
ws.merge_cells("B2:E2")

ws["B3"] = "Overview of assigned teams and waitlist (chronological priority applied)."
ws["B3"].font = Font(size=11, color="6B7280")
ws.merge_cells("B3:E3")

# KPI card helper
CARD_BG = PatternFill("solid", fgColor="FFFFFF")
CARD_BORDER = Border(left=thin, right=thin, top=thin, bottom=thin)

def kpi_card(cell, title, value, subtitle=""):
    col = cell[0]
    row = int(cell[1:])
    ws[cell] = title
    ws[cell].font = Font(size=10, bold=True, color="6B7280")
    ws[cell].alignment = Alignment(horizontal="left", vertical="top")

    ws[f"{col}{row+1}"] = value
    ws[f"{col}{row+1}"].font = Font(size=20, bold=True, color="111827")
    ws[f"{col}{row+1}"].alignment = Alignment(horizontal="left", vertical="center")

    ws[f"{col}{row+2}"] = subtitle
    ws[f"{col}{row+2}"].font = Font(size=10, color="6B7280")
    ws[f"{col}{row+2}"].alignment = Alignment(horizontal="left", vertical="bottom")

    for r in range(row, row+3):
        c = ws[f"{col}{r}"]
        c.fill = CARD_BG
        c.border = CARD_BORDER

    ws.row_dimensions[row].height = 18
    ws.row_dimensions[row+1].height = 26
    ws.row_dimensions[row+2].height = 16

# KPI row
kpi_card("B5", "Total registrations", total_regs)
kpi_card("C5", "Assigned athletes", assigned_count, "Target: 45")
kpi_card("D5", "Waitlist athletes", waitlist_count)
kpi_card("E5", "Teams created", len(final_teams), "Target: 15")

# Mini stats row
kpi_card("B9", "Olympic teams", teams_by_dist.get("olympic", 0), f"Athletes: {athletes_by_dist.get('olympic',0)}")
kpi_card("C9", "Sprint teams", teams_by_dist.get("sprint", 0), f"Athletes: {athletes_by_dist.get('sprint',0)}")
kpi_card("D9", "Competitive athletes", sum(1 for tr in teams_rows if tr["ambition"].lower()=="competitive"))
kpi_card("E9", "Fun athletes", sum(1 for tr in teams_rows if tr["ambition"].lower()=="fun"))

# -------- Traffic light ----------
target_ok = (assigned_count == 45 and len(final_teams) == 15)
partial_ok = (assigned_count <= 45 and len(final_teams) <= 15)
if target_ok:
    status_text = "ON TRACK"
    status_fill = PatternFill("solid", fgColor="DCFCE7")
    dot_color = "16A34A"
elif partial_ok:
    status_text = "NEEDS ATTENTION"
    status_fill = PatternFill("solid", fgColor="FEF9C3")
    dot_color = "CA8A04"
else:
    status_text = "ERROR"
    status_fill = PatternFill("solid", fgColor="FEE2E2")
    dot_color = "DC2626"

ws.merge_cells("B12:C12")
ws["B12"] = "Status"
ws["B12"].font = Font(size=10, bold=True, color="6B7280")
ws["B12"].alignment = Alignment(horizontal="left", vertical="center")

ws.merge_cells("D12:E12")
ws["D12"] = f"● {status_text}"
ws["D12"].font = Font(size=12, bold=True, color=dot_color)
ws["D12"].alignment = Alignment(horizontal="left", vertical="center")
for c in ("B12","C12","D12","E12"):
    ws[c].fill = status_fill
    ws[c].border = CARD_BORDER
ws.row_dimensions[12].height = 22

# -------- Waitlist widget (apply KPI-card style, NO color separation inside table) ----------
# We'll render it as 3 "cards" per distance (Olympic/Sprint), each card has Swimmer/Biker/Runner rows.
widget_title_row = 15
ws["B15"] = "Waitlist widget — earliest candidate per Distance × Role × Ambition"
ws["B15"].font = Font(size=12, bold=True, color="111827")
ws.merge_cells("B15:F15")

def draw_waitlist_card(top_left_cell, distance_label):
    # Card spans 5 rows x 5 cols: title row + header + 3 role rows
    col = top_left_cell[0]
    r0 = int(top_left_cell[1:])
    c0 = ord(col) - ord("A") + 1  # to index

    # Card background / border
    for rr in range(r0, r0+5):
        for cc in range(c0, c0+5):
            cell = ws.cell(rr, cc)
            cell.fill = CARD_BG
            cell.border = CARD_BORDER

    # Title
    ws.merge_cells(start_row=r0, start_column=c0, end_row=r0, end_column=c0+4)
    t = ws.cell(r0, c0)
    t.value = distance_label
    t.font = Font(size=11, bold=True, color="111827")
    t.alignment = Alignment(horizontal="left", vertical="center")

    # Header
    headers = ["Role", "Competitive", "Fun", "Both"]
    for i, h in enumerate(headers, start=1):
        cell = ws.cell(r0+1, c0+i)
        cell.value = h
        cell.font = Font(size=10, bold=True, color="6B7280")
        cell.alignment = Alignment(horizontal="left", vertical="center")

    # Roles
    roles = ["swimmer","biker","runner"]
    for i, role in enumerate(roles):
        ws.cell(r0+2+i, c0).value = role
        ws.cell(r0+2+i, c0).font = Font(size=10, bold=True, color="111827")
        ws.cell(r0+2+i, c0).alignment = Alignment(horizontal="left", vertical="center")

        for j, amb in enumerate(("competitive","fun","both"), start=1):
            cand = top_waitlist_candidate(distance_label.lower(), role, amb)
            email = cand["id"] if cand else ""
            cell = ws.cell(r0+2+i, c0+j)
            cell.value = email
            cell.font = Font(size=10, color="111827")
            cell.alignment = Alignment(horizontal="left", vertical="center")

    # Row heights for air
    ws.row_dimensions[r0].height = 20
    ws.row_dimensions[r0+1].height = 18
    ws.row_dimensions[r0+2].height = 20
    ws.row_dimensions[r0+3].height = 20
    ws.row_dimensions[r0+4].height = 20

# Olympic card (B17..F21)
draw_waitlist_card("B17", "Olympic")
# Sprint card (B23..F27) – keep space between cards and charts
draw_waitlist_card("B23", "Sprint")

# -------------------------
# Charts (same ranges, placed to the right and lower)
# -------------------------
bar = BarChart()
bar.type = "col"
bar.style = 10
bar.title = "Ambition by distance"
bar.y_axis.title = "Athletes"
bar.x_axis.title = "Distance"
bar_data = Reference(ws_data, min_col=3, min_row=2, max_col=5, max_row=4)
bar_cats = Reference(ws_data, min_col=2, min_row=3, max_row=4)
bar.add_data(bar_data, titles_from_data=True)
bar.set_categories(bar_cats)
bar.dataLabels = DataLabelList()
bar.dataLabels.showVal = True
bar.width = 28
bar.height = 12
ws.add_chart(bar, "B31")

pie = PieChart()
pie.title = "Assigned roles"
pie_data = Reference(ws_data, min_col=9, min_row=3, max_row=5)
pie_labels = Reference(ws_data, min_col=8, min_row=3, max_row=5)
pie.add_data(pie_data, titles_from_data=False)
pie.set_categories(pie_labels)
pie.dataLabels = DataLabelList()
pie.dataLabels.showPercent = True
pie.width = 18
pie.height = 12
ws.add_chart(pie, "G31")

# Soft background for empty space
for r in range(1, 90):
    ws.row_dimensions[r].height = ws.row_dimensions[r].height or 18
for r in ws.iter_rows(min_row=1, max_row=80, min_col=1, max_col=40):
    for c in r:
        if c.value is None and (c.fill is None or c.fill.patternType is None):
            c.fill = PatternFill("solid", fgColor="F7F9FC")

# =========================
# Teams Sheet
# =========================
ws = wb.create_sheet("Teams")
teams_headers = [
    "team_id","team_type","team_distance",
    "assigned_role","member_name","member_email",
    "gender","ambition","preferred_distance","completed_at"
]
ws.append(teams_headers)
for r in teams_rows:
    ws.append([r[h] for h in teams_headers])

style_header(ws)
apply_table_borders(ws, 1, ws.max_row, 1, len(teams_headers))
auto_width(ws)

for row_idx in range(2, ws.max_row + 1):
    dist = (ws.cell(row_idx, 3).value or "").lower()
    amb = (ws.cell(row_idx, 8).value or "").lower()
    row_fill = fill_for_distance(dist)
    for col_idx in range(1, len(teams_headers) + 1):
        ws.cell(row_idx, col_idx).fill = row_fill
    ws.cell(row_idx, 8).fill = fill_for_ambition(amb)

# =========================
# Team Summary Sheet
# =========================
ws = wb.create_sheet("Team_Summary")
sum_headers = [
    "team_id","team_type","distance","members",
    "gender_m","gender_f","gender_x",
    "competitive","fun","both",
    "swimmer","biker","runner",
]
ws.append(sum_headers)

for team_idx, team in enumerate(final_teams, start=1):
    genders = Counter(m.get("gender") for m in team["members"])
    ambitions = Counter(m.get("ambition") for m in team["members"])
    roles = Counter(team["roles"].values())
    ws.append([
        team_idx,
        team.get("team_type","AUTO"),
        team.get("distance",""),
        ", ".join(m["id"] for m in team["members"]),
        genders.get("m",0), genders.get("f",0), genders.get("x",0),
        ambitions.get("competitive",0), ambitions.get("fun",0), ambitions.get("both",0),
        roles.get("swimmer",0), roles.get("biker",0), roles.get("runner",0),
    ])

style_header(ws)
apply_table_borders(ws, 1, ws.max_row, 1, len(sum_headers))
auto_width(ws)

for row_idx in range(2, ws.max_row + 1):
    dist = (ws.cell(row_idx, 3).value or "").lower()
    row_fill = fill_for_distance(dist)
    for col_idx in range(1, len(sum_headers) + 1):
        ws.cell(row_idx, col_idx).fill = row_fill

# =========================
# Waitlist Sheet
# =========================
ws = wb.create_sheet("Waitlist")
wl_headers = [
    "completed_at","email","name","gender",
    "ambition","preferred_distance","best_role",
    "distance_hint","ambition_hint"
]
ws.append(wl_headers)
for r in waitlist_rows:
    ws.append([r[h] for h in wl_headers])

style_header(ws)
apply_table_borders(ws, 1, ws.max_row, 1, len(wl_headers))
auto_width(ws)

for row_idx in range(2, ws.max_row + 1):
    dist = (ws.cell(row_idx, 6).value or "").lower()
    amb = (ws.cell(row_idx, 5).value or "").lower()
    row_fill = fill_for_distance(dist)
    for col_idx in range(1, len(wl_headers) + 1):
        ws.cell(row_idx, col_idx).fill = row_fill
    ws.cell(row_idx, 5).fill = fill_for_ambition(amb)

# =========================
# Raw Input Sheet
# =========================
ws = wb.create_sheet("Raw_Input")
raw_headers = [
    "completed_at","email","name","gender","distance","ambition",
    "teammate_swimmer","teammate_biker","teammate_runner","remarks"
]
ws.append(raw_headers)
for r in raw_rows:
    ws.append([r[h] for h in raw_headers])

style_header(ws)
apply_table_borders(ws, 1, ws.max_row, 1, len(raw_headers))
auto_width(ws)

for row_idx in range(2, ws.max_row + 1):
    d = (ws.cell(row_idx, 5).value or "").lower()
    a = (ws.cell(row_idx, 6).value or "").lower()
    ws.cell(row_idx, 5).fill = fill_for_distance(d)
    ws.cell(row_idx, 6).fill = fill_for_ambition(a)

# Save
wb.save(OUTPUT_XLSX)
print(f"✅ Dashboard-style Excel created: {OUTPUT_XLSX}")

✅ Dashboard-style Excel created: /content/zc3_team_assignment_result.xlsx
